# `fuzzy_search` function tests

`fuzzy_search(query)` returns:
- a **string** if an exact match is found
- a **list of strings** (up to 10) if no exact match is found

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

from scripts.utils.fuzzy_search import fuzzy_search

## Exact match

In [2]:
fuzzy_search("Amanita muscaria")

'Amanita muscaria'

## Misspelling

In [3]:
fuzzy_search("Amanita muscara")

['Amanita muscaria', 'Amanitaria muscaria']

## Partial name

In [4]:
fuzzy_search("Amanita musc")

['Amanita curtipes',
 'Amanita parvivirginea',
 'Amanita squamosa',
 'Amanita praelongipes',
 'Amanita rubrovolvata',
 'Amanita alliacea',
 'Amanita subglobosa',
 'Amanita luteofolia',
 'Amanita pseudovalens',
 'Amanita sepiacea']

## Common name

In [5]:
fuzzy_search("fly agaric")

[]

## No match

In [6]:
fuzzy_search("completelymadeupname")

[]

# GBIF Fuzzy Search & Autocomplete

This part of the notebook demonstrates three GBIF API endpoints useful for resolving messy or partial species names:

| Endpoint | Purpose |
|---|---|
| `GET /v1/species/match` | Fuzzy match a name against the GBIF Backbone Taxonomy |
| `GET /v1/species/suggest` | Autocomplete — returns ranked candidates for a partial string |
| `GET /v1/species/search` | Full-text search across all name fields |

All examples use **_Amanita muscaria_** (fly agaric) as the reference species.

In [7]:
import requests
import pandas as pd

GBIF_BASE = "https://api.gbif.org/v1"

---
## 1. Fuzzy Name Matching — `species/match`

The match endpoint scores a name against the GBIF Backbone and returns the best hit along with a `matchType`:

- `EXACT` — character-for-character match
- `FUZZY` — best approximate match (handles misspellings, alternate authorship, etc.)
- `HIGHERRANK` — only a genus / family match was found
- `NONE` — no match

Setting `strict=false` (the default) allows fuzzy matching.  
Setting `strict=true` restricts results to `EXACT` matches only.

In [8]:
def species_match(name: str, strict: bool = False) -> dict:
    resp = requests.get(
        f"{GBIF_BASE}/species/match",
        params={"name": name, "strict": str(strict).lower()},
    )
    resp.raise_for_status()
    return resp.json()

In [9]:
# Exact name — should return matchType EXACT
result = species_match("Amanita muscaria")
print(f"Input      : Amanita muscaria")
print(f"Match type : {result['matchType']}")
print(f"Accepted   : {result.get('species')}")
print(f"Status     : {result.get('status')}")
print(f"Usage key  : {result.get('usageKey')}")
print(f"Confidence : {result.get('confidence')}")

Input      : Amanita muscaria
Match type : EXACT
Accepted   : Amanita muscaria
Status     : ACCEPTED
Usage key  : 8168319
Confidence : 97


In [10]:
# Misspelled name — fuzzy match should still resolve it
misspellings = [
    "Amanita muscara",      # dropped 'i'
    "Amanita muscarria",    # doubled 'r'
    "amanita muscaria",     # lowercase genus
    "Amanita Muscaria",     # capitalised epithet
    "Amanata muscaria",     # 'a' instead of 'i' in genus
]

rows = []
for name in misspellings:
    r = species_match(name)
    rows.append({
        "Input": name,
        "Match type": r.get("matchType"),
        "Resolved name": r.get("species"),
        "Confidence": r.get("confidence"),
        "Status": r.get("status"),
    })

pd.DataFrame(rows)

,Input,Match type,Resolved name,Confidence,Status
0,Amanita muscara,FUZZY,Amanita muscaria,93,ACCEPTED
1,Amanita muscarria,FUZZY,Amanita muscaria,95,ACCEPTED
2,amanita muscaria,EXACT,Amanita muscaria,97,ACCEPTED
3,Amanita Muscaria,NONE,None,100,None
4,Amanata muscaria,FUZZY,Amanita muscaria,80,ACCEPTED


---
## 2. Autocomplete / Suggest — `species/suggest`

The suggest endpoint is designed for search-box autocomplete.  
It ranks candidates by usage frequency and returns up to `limit` results.

In [11]:
def species_suggest(q: str, limit: int = 10, rank: str = None) -> list[dict]:
    params = {"q": q, "limit": limit}
    if rank:
        params["rank"] = rank
    resp = requests.get(f"{GBIF_BASE}/species/suggest", params=params)
    resp.raise_for_status()
    return resp.json()

In [12]:
# Partial prefix — simulates a user typing into a search box
suggestions = species_suggest("Amanita musc", limit=10)

pd.DataFrame([
    {
        "Canonical name": s.get("canonicalName"),
        "Rank": s.get("rank"),
        "Status": s.get("status"),
        "Kingdom": s.get("kingdom"),
        "Usage key": s.get("key"),
    }
    for s in suggestions
])

,Canonical name,Rank,Status,Kingdom,Usage key
0,Amanita muscoides,SPECIES,ACCEPTED,Fungi,5452468
1,Amanita muscaria,SPECIES,ACCEPTED,Fungi,8168319
2,Amanita muscaria,SPECIES,SYNONYM,Fungi,3329049
3,Amanitaria muscaria,SPECIES,SYNONYM,Fungi,3301084
4,Amanita muscaria,SPECIES,SYNONYM,Fungi,8125569
5,Amanita muscaria flavivolvata,VARIETY,ACCEPTED,Fungi,7874844
6,Amanita muscaria lutea,VARIETY,ACCEPTED,Fungi,11183558
7,Amanita muscaria muscaria,VARIETY,ACCEPTED,Fungi,8363881
8,Amanita muscaria americana,VARIETY,ACCEPTED,Fungi,7707751
9,Amanita muscaria formosa,SUBSPECIES,SYNONYM,Fungi,11305013


In [13]:
# Restrict to SPECIES rank to filter out genera / families
species_only = species_suggest("Amanita musc", limit=10, rank="SPECIES")

pd.DataFrame([
    {
        "Canonical name": s.get("canonicalName"),
        "Status": s.get("status"),
        "Family": s.get("family"),
        "Usage key": s.get("key"),
    }
    for s in species_only
])

,Canonical name,Status,Family,Usage key
0,Amanita muscoides,ACCEPTED,Amanitaceae,5452468
1,Amanita muscaria,ACCEPTED,Amanitaceae,8168319
2,Amanita muscaria,SYNONYM,Amanitaceae,3329049
3,Amanitaria muscaria,SYNONYM,Amanitaceae,3301084
4,Amanita muscaria,SYNONYM,Amanitaceae,8125569


---
## 3. Full-text Search — `species/search`

The search endpoint supports free-text queries (`q`) against all name fields — useful when the user supplies a common name, synonym, or partial scientific name.  
Results can be filtered by `rank`, `status`, `kingdom`, etc.

In [14]:
def species_search(q: str, rank: str = None, status: str = None, limit: int = 10) -> dict:
    params = {"q": q, "limit": limit}
    if rank:
        params["rank"] = rank
    if status:
        params["status"] = status
    resp = requests.get(f"{GBIF_BASE}/species/search", params=params)
    resp.raise_for_status()
    return resp.json()

In [15]:
# Search by scientific name
results = species_search("Amanita muscaria", rank="SPECIES", limit=8)
print(f"Total hits: {results['count']}")

pd.DataFrame([
    {
        "Canonical name": r.get("canonicalName"),
        "Authorship": r.get("authorship"),
        "Status": r.get("taxonomicStatus"),
        "Accepted name": r.get("accepted"),
        "Usage key": r.get("key"),
    }
    for r in results.get("results", [])
])

Total hits: 183


,Canonical name,Authorship,Status,Accepted name,Usage key
0,Amanita muscaria,None,ACCEPTED,None,206104232
1,Amanita muscaria,None,SYNONYM,"Amanita muscaria (L.) Lam., 1783",163302243
2,Amanita muscaria,None,ACCEPTED,None,163998449
3,Amanita muscaria,None,ACCEPTED,None,165611965
4,Amanita muscaria,None,SYNONYM,Amanita muscaria (L.) Lam.,209816399
5,Amanita muscaria,Aubl.,ACCEPTED,None,184463523
6,Amanita muscaria,(L.) Lam.,ACCEPTED,None,180077016
7,Amanita muscaria,None,ACCEPTED,None,142409633


In [16]:
# Search by common (vernacular) name — fly agaric
results_common = species_search("fly agaric", rank="SPECIES", limit=5)
print(f"Total hits: {results_common['count']}")

pd.DataFrame([
    {
        "Canonical name": r.get("canonicalName"),
        "Status": r.get("taxonomicStatus"),
        "Kingdom": r.get("kingdom"),
        "Usage key": r.get("key"),
    }
    for r in results_common.get("results", [])
])

Total hits: 22


,Canonical name,Status,Kingdom,Usage key
0,Amanita muscaria,ACCEPTED,Fungi,205177255
1,Amanita muscaria,SYNONYM,Fungi,160781717
2,Amanita muscaria,ACCEPTED,Fungi,202600701
3,Amanita muscaria,ACCEPTED,Fungi,103725206
4,Amanita muscaria,ACCEPTED,None,165611965


---
## 4. Putting it Together — a Simple Name Resolver

This helper tries `species/match` first (fast, single result).  
If confidence is below a threshold it falls back to `species/suggest` and returns the top candidate.

In [17]:
def resolve_species_name(name: str, confidence_threshold: int = 80) -> dict | None:
    """
    Returns a dict with keys: resolved_name, match_type, confidence, usage_key.
    Returns None if no match is found by either endpoint.
    """
    match = species_match(name)

    if match.get("matchType") != "NONE" and match.get("confidence", 0) >= confidence_threshold:
        return {
            "resolved_name": match.get("species"),
            "match_type": match.get("matchType"),
            "confidence": match.get("confidence"),
            "usage_key": match.get("usageKey"),
            "source": "match",
        }

    # Fallback: suggest
    suggestions = species_suggest(name, limit=1, rank="SPECIES")
    if suggestions:
        top = suggestions[0]
        return {
            "resolved_name": top.get("canonicalName"),
            "match_type": "SUGGEST_FALLBACK",
            "confidence": None,
            "usage_key": top.get("key"),
            "source": "suggest",
        }

    return None

In [18]:
test_names = [
    "Amanita muscaria",
    "Amanita muscara",
    "Amanita musc",
    "fly agaric",
    "completelymadeupname",
]

rows = []
for name in test_names:
    result = resolve_species_name(name)
    if result:
        rows.append({"Input": name, **result})
    else:
        rows.append({"Input": name, "resolved_name": None, "match_type": "NONE",
                     "confidence": None, "usage_key": None, "source": None})

pd.DataFrame(rows)

,Input,resolved_name,match_type,confidence,usage_key,source
0,Amanita muscaria,Amanita muscaria,EXACT,97.0,8168319.0,match
1,Amanita muscara,Amanita muscaria,FUZZY,93.0,8168319.0,match
2,Amanita musc,None,HIGHERRANK,94.0,6005964.0,match
3,fly agaric,None,NONE,NaN,NaN,None
4,completelymadeupname,None,NONE,NaN,NaN,None
